# From linear classification to neural networks

**Lab 5 · Building and fitting a neural network**

In Labs 3 and 4, you implemented predictors and classifiers, fitted their parameters,
and examined their empirical losses. Here you will do the same for a neural network.
We supply the familiar affine classifier on a new dataset. **You will implement the
network, use automatic differentiation to obtain its gradients, adapt gradient
descent, and plot and interpret the results.**

Use **Google Colab** and work from top to bottom; no GPU is needed. At each exercise,
write your code in the marked cell before opening **Show answer**. The answers include
complete solutions if you get stuck. Opening an answer does not execute its code:
copy it into that exercise's code cell when needed, then continue.

New PyTorch operations are explained before you need them. The data generation,
affine comparison and technical machinery for drawing boundaries are supplied.
The final extension is optional. This notebook contains the complete Week 5 lab.

## 1. Generate and visualise the data

We generate **100 observations**, arranged in four Gaussian clouds. Each cloud has 25 observations, with independent coordinate noise of standard deviation $0.45$.

| Cloud centre | Class | Observations |
|---|---:|---:|
| $(-1,-1)$ | 0 | 25 |
| $(1,1)$ | 0 | 25 |
| $(-1,1)$ | 1 | 25 |
| $(1,-1)$ | 1 | 25 |

Diagonally opposite clouds share a label. This is a noisy version of the XOR arrangement from the lectures: the inputs are real-valued, and each label records the class assigned to the cloud that generated the observation. Blue circles show class 0 and orange triangles show class 1, as in Lab 4.

The tensor `X` contains the two input coordinates and has shape `(N, 2)`. The labels in `y` have shape `(N, 1)`. Run the supplied cell to create and visualise the data.

In [ ]:
import torch as t
import matplotlib.pyplot as plt
t.manual_seed(1)  # Fix the random seed for this notebook.

N = 100
# Four Gaussian clouds in an XOR arrangement: 25 observations per cloud.
# spread is the standard deviation of each coordinate, not a variance.
spread = 0.45
x_class_0 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([-1.0, -1.0]),
    t.randn(N // 4, 2) * spread + t.tensor([ 1.0,  1.0])
], 0)
x_class_1 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([-1.0,  1.0]),
    t.randn(N // 4, 2) * spread + t.tensor([ 1.0, -1.0])
], 0)
X = t.cat([x_class_0, x_class_1], 0)              # Shape: (100, 2).
y = t.cat([t.zeros(N // 2, 1), t.ones(N // 2, 1)], 0)  # Shape: (100, 1).

# Show observed classes using the same colours and markers as in Lab 4.
plt.figure()
plt.scatter(x=x_class_0[:, 0], y=x_class_0[:, 1], color='tab:blue', label='Class 0')
plt.scatter(x=x_class_1[:, 0], y=x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.axis('equal')
plt.legend()
plt.show()


# Supplied: remember the random-generator state immediately after creating the data.
# The parameter exercise restores it, so rerunning that exercise is reproducible.
parameter_initial_state = t.get_rng_state().clone()

## 2. Reuse the linear classifier from Lab 4

Look at the four clouds. **Could a single straight decision boundary separate the four centres according to their labels?** Keep your prediction in mind while running the model.

The code below supplies the completed calculations from Lab 4: the stable empirical logistic loss, its gradient and the number of classification mistakes. You do not need to implement them again. Recall why we use `logsigmoid` to evaluate the logarithms directly.

As in Lab 4, we include the intercept using a column of ones. Here that design matrix is called `X_affine`, while `X` keeps the original coordinates for the later network. The functions are prefixed with `affine_` to keep the two calculations separate. Their mathematical definitions are unchanged.

The coefficient column contains $(b,w_1,w_2)$, and the scores are `X_affine @ theta`. The predicted label is 1 when the sigmoid of the score is greater than 0.5, and 0 otherwise, including at equality. This is equivalent to testing whether the score is strictly positive.

In [ ]:
# Supplied: the completed functions from Lab 4, with affine-specific names.
X_affine = t.cat([t.ones(N, 1), X], dim=1)  # Shape (N, 3), including the intercept.

def affine_loss(theta):
    scores = X_affine @ theta
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

def affine_grad(theta):
    scores = X_affine @ theta
    probabilities = t.sigmoid(scores)
    return X_affine.mT @ (probabilities - y)

def affine_mistakes(theta):
    probabilities = t.sigmoid(X_affine @ theta)
    predicted_labels = (probabilities > 0.5).to(y.dtype)
    return int((predicted_labels != y).sum().item())

### 2.1. Run gradient descent

Run the supplied update loop. We use Lab 4's zero initialisation and learning rate `0.01`, now allowing **1,000 updates**. Every update uses the full dataset.

As before, record the initial loss and mistake count, then record them again **after** each update. Entry `step` in each history therefore describes exactly `step` completed updates. We also keep copies of the parameters at selected steps for the boundary plot.

In [ ]:
affine_theta = t.zeros(3, 1)
affine_learning_rate = 0.01
num_updates = 1000
affine_loss_history = [affine_loss(affine_theta).item()]
affine_error_history = [affine_mistakes(affine_theta)]
affine_checkpoints = {}

for step in range(1, num_updates + 1):
    affine_theta = affine_theta - affine_learning_rate * affine_grad(affine_theta)
    affine_loss_history.append(affine_loss(affine_theta).item())
    affine_error_history.append(affine_mistakes(affine_theta))

    # Supplied: save independent copies for the later boundary plot.
    if step in (1, 10, 100, num_updates):
        affine_checkpoints[step] = affine_theta.clone()

print('Updates   Empirical logistic loss   Mistakes')
for step in (0, 1, 10, 100, num_updates):
    print(f'{step:7d}   {affine_loss_history[step]:23.6f}   {affine_error_history[step]:8d}')

### 2.2. Inspect the curves and the boundaries

Run the supplied plotting cells. These are the same quantities we examined in Lab 4:

- the summed empirical logistic loss;
- the loss divided by $\log 2$, alongside the number of mistakes;
- the decision boundaries at successive updates.

The division by $\log 2$ is only for display: **the updates use the original, unscaled loss**. Each mistake contributes at least $\log 2$ to that loss, so the scaled loss bounds the mistake count from above.

All affine boundaries have the form $b+w_1x_1+w_2x_2=0$. For these saved coefficients, $w_2$ is nonzero, so we can draw the lines using $x_2=-(b+w_1x_1)/w_2$, just as in Lab 4. This plotting code is supplied; focus on interpreting its output.

In [ ]:
import math

updates = range(num_updates + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(updates, affine_loss_history, color='0.35')
axes[0].set_title('Affine model: empirical logistic loss')
axes[0].set_ylabel('Summed loss')
axes[0].set_ylim(0, 75)  # Keep the small change in perspective; the table gives exact values.

axes[1].plot(updates, [value / math.log(2) for value in affine_loss_history],
             color='tab:purple', label=r'Loss / $\log 2$')
axes[1].plot(updates, affine_error_history, color='tab:green', label='Mistakes')
axes[1].set_title('Affine model: loss and mistakes')
axes[1].set_ylabel('Value')
axes[1].set_ylim(0, 110)
axes[1].legend()
for ax in axes:
    ax.set_xlabel('Completed updates')
    ax.grid(alpha=0.2)
plt.show()

In [ ]:
# The line-plotting calculation from Lab 4, using the new observations.
x1_limits = (X[:, 0].min().item() - 0.5, X[:, 0].max().item() + 0.5)
x2_limits = (X[:, 1].min().item() - 0.5, X[:, 1].max().item() + 0.5)
x1_line = t.tensor(x1_limits)

fig, ax = plt.subplots(figsize=(8.5, 5.2), constrained_layout=True)
plt.scatter(x_class_0[:, 0], x_class_0[:, 1], color='tab:blue', label='Class 0')
plt.scatter(x_class_1[:, 0], x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
for step, style in zip((1, 10, 100, num_updates), (':', '--', '-.', '-')):
    b, w1, w2 = affine_checkpoints[step][:, 0]
    x2_line = -(b + w1 * x1_line) / w2
    plt.plot(x1_line, x2_line, color='black', linestyle=style,
             label=f'After {step} update' + ('s' if step != 1 else ''))
plt.xlim(x1_limits)
plt.ylim(x2_limits)
plt.gca().set_aspect('equal')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Affine decision boundaries')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.show()

### Pause and interpret: what has gradient descent achieved?

1. Compare the initial and final empirical losses and mistake counts. Has the affine model fitted the observations well? The loss curve looks almost flat: does this mean that no updates have taken place? Use the printed values to read changes that are too small to see in the curves.
2. What happens to the boundary between updates 1, 100 and 1,000? Could more updates turn it into a boundary with several segments?
3. Why do the loss and the number of mistakes not necessarily decrease together?

Write down your answers before opening the explanation.

<details>
<summary>Show answer — interpreting the affine fit</summary>

**1. The loss decreases very little, and many mistakes remain.** With the supplied seed, the loss changes from approximately $69.3147$ to $69.3083$, while the mistake count changes from 50 to 47. The loss has already essentially stopped changing by update 100. The fitted model still misclassifies almost half of these observations.

The curve looks flat because the total decrease is only about $0.00645$. Initially, all coefficients are zero, so every score is zero and every predicted probability is $1/2$. The summed empirical loss is therefore $N\log 2\approx69.3147$.

For this nearly symmetric XOR sample, the fitted affine scores remain close to zero: a single straight decision boundary cannot capture the main XOR pattern. Small sample asymmetries allow a slight improvement, including a few changed class predictions, but the probabilities remain near $1/2$. The empirical logistic loss is convex in the affine coefficients, so this behaviour is not due to a suboptimal local minimum. It illustrates a limitation of the affine model on these data.

**2. Updates change the coefficients of a straight line.** The boundary initially rotates and shifts; the lines at 100 and 1,000 updates almost coincide. Further updates can change its slope and intercept, but an affine score always has a straight boundary when its feature weights are nonzero. A single straight line cannot separate the four cloud centres according to their XOR labels. We need a richer family of scores to express a different shape.

**3. The two quantities measure different things.** The logistic loss depends on the probabilities assigned to the observed labels. The mistake count only checks which side of the classification threshold each observation falls on. Probabilities can change without changing any labels, and an improvement in total loss can occur even when the number of mistakes increases.

The 47 mistakes are the outcome of fitting the logistic loss in this run. They are not a claim that 47 is the smallest mistake count attainable by every possible straight boundary.

</details>

## 3. Can we do better with a neural network?

Keep the **same observations, labels and empirical logistic loss**, and change the
model that produces the scores. In the lectures, the hidden weights and biases
made the feature functions adjustable. You will now implement that model and fit
all of its parameters.

In Lab 4, we wrote the gradient formula ourselves. Here, PyTorch's **automatic
differentiation** will calculate the derivatives for us. You will still implement
the gradient-descent updates. The next lecture explains **backpropagation**, the
algorithm behind this derivative calculation.

### Reuse the empirical logistic loss

Our Lab 4 loss function calculated the affine scores internally. The supplied
`loss_from_scores` instead takes already-computed scores and evaluates the same
empirical loss, using the labels in `y`:

$$\mathcal L_{\mathrm{log}} = \sum_{i=1}^N
\ell_{\mathrm{log}}(\sigma(s_i),y_i).$$

We can use it with either affine or neural-network scores. Recall Lab 4's numerical
stability issue: we again use `logsigmoid` to evaluate the logarithms directly.
**Pass scores to this function without first applying a sigmoid.** You do not need
to implement this familiar loss again.

In [ ]:
def loss_from_scores(scores):
    # scores contains classification scores, not probabilities or weights.
    # Evaluate sum_i ell_log(sigmoid(score_i), y_i), using the fixed labels y.
    # logsigmoid computes the logarithms directly, without forming probabilities.
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

### 3.1. From the architecture to parameter tensors

Use **two input coordinates, 100 hidden neurons and one scalar output**. In the
notes' notation, $L=2$, $m_0=2$, $m_1=100$ and $m_2=1$. For one input row,

$$
\begin{aligned}
\boldsymbol a_1(\boldsymbol x)&=\boldsymbol x\boldsymbol W_1+\boldsymbol b_1,\\
\boldsymbol\phi_1(\boldsymbol x)&=\operatorname{\mathbf{ReLU}}(\boldsymbol a_1(\boldsymbol x)),\\
f_{\boldsymbol\theta}(\boldsymbol x)&=\boldsymbol\phi_1(\boldsymbol x)\boldsymbol W_2+\boldsymbol b_2.
\end{aligned}
$$

The subscript identifies the layer. The column $\boldsymbol\theta$ collects all
weights and biases mathematically; our code will store them in four tensors.
`X` has shape `(N, 2)` and contains only the original coordinates. Unlike
`X_affine`, it needs no extra column of ones: biases are stored separately.

`t.randn(rows, columns, requires_grad=True)` creates a matrix of random initial
values and asks PyTorch to track computations involving it. This will let us
differentiate a loss with respect to its entries later; it does not compute any
derivatives yet. With one size argument, `t.randn(size, requires_grad=True)`
creates a one-dimensional tensor. We use one-dimensional tensors for the biases:
adding a bias vector to a matrix adds it to each row (**broadcasting**).

**Exercise 1 — initialise the parameters.**

1. Work out the shapes of `W1`, `b1`, `W2` and `b2` from the matrix products above.
2. Create them using `t.randn(..., requires_grad=True)`. Use the dimension variables
   supplied below, and create them in the order **`b1`, `W1`, `b2`, `W2`** so that
   your random draws match the reference experiment.
3. Print their shapes. How many adjustable scalar parameters are there altogether?

The supplied first line restores the saved random-generator state, making this
cell repeatable. The final lines keep independent copies of the initial parameters
for later resets: `detach()` excludes a copy from gradient tracking and `clone()`
gives it separate storage. Keep these lines in this cell, below your initialisations.

In [ ]:
t.set_rng_state(parameter_initial_state)  # Supplied: reproduce the initial random draws.
input_features = 2
hidden_features = 100
output_features = 1

# YOUR CODE: create b1, W1, b2 and W2, then print shapes and count scalar parameters.

# Supplied: save the initial values immediately after creating the parameters.
initial_b1 = b1.detach().clone()
initial_W1 = W1.detach().clone()
initial_b2 = b2.detach().clone()
initial_W2 = W2.detach().clone()

<details>
<summary>Show answer — parameter shapes and initialisation</summary>

`W1` has shape `(2, 100)`, `b1` has shape `(100,)`, `W2` has shape
`(100, 1)` and `b2` has shape `(1,)`. The total is
$2\times100+100+100\times1+1=401$ scalar parameters.
The notes' bias rows are represented by one-dimensional tensors in this code.

```python
t.set_rng_state(parameter_initial_state)  # Supplied: reproduce the initial random draws.
input_features = 2
hidden_features = 100
output_features = 1

b1 = t.randn(hidden_features, requires_grad=True)
W1 = t.randn(input_features, hidden_features, requires_grad=True)
b2 = t.randn(output_features, requires_grad=True)
W2 = t.randn(hidden_features, output_features, requires_grad=True)

for name, parameter in [('W1', W1), ('b1', b1), ('W2', W2), ('b2', b2)]:
    print(name, tuple(parameter.shape))
print('Scalar parameters:', W1.numel() + b1.numel() + W2.numel() + b2.numel())

# Supplied: save the initial values immediately after creating the parameters.
initial_b1 = b1.detach().clone()
initial_W1 = W1.detach().clone()
initial_b2 = b2.detach().clone()
initial_W2 = W2.detach().clone()
```

</details>

### 3.2. Implement the network calculation

The equations above describe one observation. Applying them to the whole matrix
`X` performs the same calculation for every row, using the same parameters.
`t.relu(A)` applies $\max\{0,a\}$ separately to every entry of `A`; it implements
the componentwise ReLU from the notes.

**Exercise 2 — write `net(X)`.** Translate the three equations into a Python
function with intermediate variables `a1` and `phi1`, returning `scores`.
Use the four tensors you have just created; do not draw new parameters inside
the function. Before running it, predict the shapes of all three intermediate
results for the dataset. What should happen if you pass only one input row as
`X[:1]`? This slice keeps a matrix with shape `(1, 2)`; `X[0]` would instead
return a one-dimensional tensor.

The hidden activations are values of learned feature functions. The final output
is a classification **score**, not a probability or a predicted label.

In [ ]:
# YOUR CODE: define net(X).

<details>
<summary>Show answer — the network calculation</summary>

`a1` and `phi1` have shape `(N, 100)`; `scores` has shape `(N, 1)`.
Passing one row gives the same calculation with `N = 1`. The variables contain
values for the inputs passed to the function; the parameter tensors are reused.

```python
def net(X):
    a1 = X @ W1 + b1
    phi1 = t.relu(a1)
    scores = phi1 @ W2 + b2
    return scores
```

</details>

In [ ]:
# Supplied checks: one score per observation, also for a single-row input.
assert net(X).shape == y.shape
assert net(X[:1]).shape == (1, 1)
print('Dataset output shape:', tuple(net(X).shape))

### 3.3. Evaluate the empirical loss and mistakes

**Exercise 3 — evaluate your initial model.**

1. Calculate its scores and empirical logistic loss using `loss_from_scores`.
2. Write `mistakes_from_scores(scores)`, returning the number of incorrect labels
   as a Python integer. Like the loss helper, it should use the fixed labels `y`.
   Reuse Lab 4's decision rule: class 1 for a strictly positive score and class 0
   otherwise, including when the score is zero.
3. Print the loss and the number of mistakes before fitting.

Use `.item()` to obtain the number in a scalar tensor; use `int(...)` if needed
for a Python integer. Do not add a sigmoid to the network output for the loss.
To repeat this initial evaluation after fitting, rerun Exercise 1 first.

In [ ]:
# YOUR CODE: define mistakes_from_scores, then evaluate the initial model.

<details>
<summary>Show answer — initial loss and mistakes</summary>

Compare each predicted label with its observed label and count disagreements.
The supplied initialisation gives loss about $154.819$ and 30 mistakes. A random
network can already classify some observations correctly; these are its results
before any fitting updates.

```python
def mistakes_from_scores(scores):
    predicted_labels = (scores > 0).to(y.dtype)
    return int((predicted_labels != y).sum().item())

scores = net(X)
L = loss_from_scores(scores)
print('Initial empirical loss:', L.item())
print('Initial mistakes:', mistakes_from_scores(scores))
```

</details>

In [ ]:
# Supplied checks on artificial scores, including ties at zero.
assert mistakes_from_scores(t.zeros_like(y)) == 50
assert mistakes_from_scores(2 * y - 1) == 0
print('Mistake-count checks passed.')

### 3.4. Ask PyTorch for the derivatives

Here is a small example separate from the network. Predict the two partial
derivatives of $Q(u)=u_0^2+u_1^2$ at $u=(1,-2)$, then run the cell.
`Q.backward()` computes the derivatives and stores them in `u.grad`.

In [ ]:
u = t.tensor([1.0, -2.0], requires_grad=True)
Q = (u ** 2).sum()
Q.backward()
print('u:', u)
print('Gradient of Q:', u.grad)

The gradient is $(2,-4)$ and `u` has not changed. For a network parameter tensor,
each `.grad` entry is similarly the derivative of the scalar empirical loss with
respect to the parameter entry in the same position. For example, `W1.grad[0, 0]`
corresponds to `W1[0, 0]`. Before a gradient is stored, `.grad` is `None`.

**`backward()` does not update parameters. It adds to any gradients already stored.**
The gradient of a sum is the sum of the gradients: we could calculate losses for
separate observations and call `backward()` for each, holding parameters fixed.
The accumulated result would be the gradient needed for one full-dataset update.
This is useful when data must be processed in smaller pieces.

Our empirical loss already sums over the whole small dataset using tensor
operations. One call computes its full gradient, without a Python loop over
observations. We therefore clear old gradients before calculating the next ones.
**Calculating a new loss does not clear `.grad`.** PyTorch uses 0 as the ReLU
multiplier at input 0; its ordinary derivative remains undefined there.

**Exercise 4 — compute the network's gradients.** After the supplied clearing
code, calculate a fresh loss from your network and ask PyTorch to differentiate
it. Print each parameter's shape and its gradient's shape.

The derivatives refer to the current parameter values. If you want to examine
the initial parameters again after fitting, rerun Exercise 1 first.

Explain why the shapes match. If an entry of `W1.grad` is positive, should the
corresponding entry of `W1` increase or decrease in a gradient-descent update
with a positive learning rate?

In [ ]:
# Supplied: make this gradient demonstration repeatable.
for parameter in (b1, W1, b2, W2):
    if parameter.grad is not None:
        parameter.grad.zero_()

# YOUR CODE: calculate a fresh loss, differentiate it, and print the shapes.

<details>
<summary>Show answer — derivatives of the network loss</summary>

Each loss derivative corresponds to one parameter entry, so the shapes agree.
A positive derivative gives a negative change in that weight because gradient
descent subtracts the learning rate times the derivative. All these derivatives
are evaluated before any parameters are changed.

```python
# Supplied: make this gradient demonstration repeatable.
for parameter in (b1, W1, b2, W2):
    if parameter.grad is not None:
        parameter.grad.zero_()

L = loss_from_scores(net(X))
L.backward()
for name, parameter in [('W1', W1), ('b1', b1), ('W2', W2), ('b2', b2)]:
    print(name, tuple(parameter.shape), tuple(parameter.grad.shape))
```

</details>

### 3.5. Implement gradient descent

You already know the update rule: subtract the learning rate times each current
partial derivative from the corresponding parameter. Two pieces of Python/PyTorch
syntax let us apply it to tensors that require gradients.

**`with` is a Python keyword introducing a block with a temporary context.**
`with t.no_grad():` turns off gradient tracking for the indented instructions.
They run normally; after the block, the previous tracking setting is restored.
We have already computed the derivatives and do not need to differentiate the
parameter-update operation itself. This context does not clear stored gradients.

**`.add_(...)` changes a tensor in place.** For example, the following cell applies
one update to the separate toy variable `u` above. Then `u.grad.zero_()` clears
the stored derivatives without changing `u`. Run the toy derivative cell again
first if you want to repeat this example from its initial values.

In [ ]:
with t.no_grad():
    u.add_(-0.1 * u.grad)
u.grad.zero_()
print('After the toy update:', u)  # (0.8, -1.6)
print('Cleared gradient:', u.grad)

The next cell supplies the experiment's reset and plotting setup. Run it before
the fitting exercise. `restart_network()` restores the initial parameter values
saved in Exercise 1 and clears any stored gradients. `.copy_(...)` copies values
into an existing tensor in place. It does not draw a new initialisation.

The grid is only for pictures of decision boundaries. It contains no extra
observations used for fitting and does not alter the empirical loss.

In [ ]:
# Supplied experiment infrastructure.
def restart_network():
    with t.no_grad():
        b1.copy_(initial_b1)
        W1.copy_(initial_W1)
        b2.copy_(initial_b2)
        W2.copy_(initial_W2)
    for parameter in (b1, W1, b2, W2):
        if parameter.grad is not None:
            parameter.grad.zero_()

# Supplied plotting grid; reproducing this construction is not an exercise.
x1_limits = (X[:, 0].min().item() - 0.5, X[:, 0].max().item() + 0.5)
x2_limits = (X[:, 1].min().item() - 0.5, X[:, 1].max().item() + 0.5)
x1_grid, x2_grid = t.meshgrid(t.linspace(*x1_limits, 250),
                            t.linspace(*x2_limits, 250), indexing='xy')
X_grid = t.stack([x1_grid, x2_grid], dim=-1).reshape(-1, 2)

**Exercise 5 — adapt gradient descent to all four parameter tensors.**

Complete the marked parts of the cell below. Use the full dataset for each of
1,000 updates, with learning rate `0.0001`. We use a different learning rate from
the affine model because its parameterisation differs; we are comparing the
resulting fits, not measuring which algorithm needs fewer updates.

1. Create `loss_history` and `error_history`, each starting with the initial
   measurement. Store ordinary Python numbers, not tensors with tracked computations.
2. At each step, compute a fresh empirical loss and all its gradients.
3. Inside the supplied `no_grad` block, update **all four** parameter tensors using
   those gradients. Then recompute scores and append the new loss and mistake count.
4. Clear the stored gradients before the next iteration.

All derivatives must be calculated at the same current parameters, before any
parameters are updated. Entry `step` in each history must describe exactly
`step` completed updates, so each finished history has 1,001 entries.

The reset at the start makes reruns repeat the experiment. Keep the supplied
snapshot lines where they are: they store scores **after** the selected updates,
before later updates change the network again. The snapshots are only for plots.

In [ ]:
restart_network()  # Supplied: every run begins at the saved initial parameters.
learning_rate = 0.0001
num_updates = 1000
boundary_steps = (1, 100, num_updates)
boundary_scores = {}

# YOUR CODE: initialise loss_history and error_history, including update 0.

for step in range(1, num_updates + 1):
    # YOUR CODE: evaluate a fresh loss and compute its gradients.

    with t.no_grad():
        # YOUR CODE: update b1, W1, b2 and W2 using their gradients.
        # YOUR CODE: recompute scores and append loss and mistakes to the histories.

        # Supplied: save independent score values for the later boundary figures.
        if step in boundary_steps:
            boundary_scores[step] = net(X_grid).reshape(x1_grid.shape).clone()

    # YOUR CODE: clear the stored gradients before the next iteration.

<details>
<summary>Show answer — the gradient-descent loop</summary>

The algorithm is the same as in Lab 4. The parameters now occupy four tensors,
and `backward()` calculates their derivatives. Measurements below use the
updated parameters; the next iteration constructs a fresh loss for differentiation.
We clear all four gradients, not the parameters themselves.

```python
restart_network()  # Supplied: every run begins at the saved initial parameters.
learning_rate = 0.0001
num_updates = 1000
boundary_steps = (1, 100, num_updates)
boundary_scores = {}

with t.no_grad():
    scores = net(X)
    loss_history = [loss_from_scores(scores).item()]
    error_history = [mistakes_from_scores(scores)]

for step in range(1, num_updates + 1):
    L = loss_from_scores(net(X))
    L.backward()

    with t.no_grad():
        b1.add_(-learning_rate * b1.grad)
        W1.add_(-learning_rate * W1.grad)
        b2.add_(-learning_rate * b2.grad)
        W2.add_(-learning_rate * W2.grad)

        scores = net(X)
        loss_history.append(loss_from_scores(scores).item())
        error_history.append(mistakes_from_scores(scores))

        # Supplied: save independent score values for the later boundary figures.
        if step in boundary_steps:
            boundary_scores[step] = net(X_grid).reshape(x1_grid.shape).clone()

    b1.grad.zero_()
    W1.grad.zero_()
    b2.grad.zero_()
    W2.grad.zero_()
```

</details>

In [ ]:
# Supplied checks and summary of your recorded results.
assert len(loss_history) == num_updates + 1
assert len(error_history) == num_updates + 1
assert set(boundary_scores) == set(boundary_steps)
print('Updates   Empirical logistic loss   Mistakes')
for step in (0, 1, 10, 100, num_updates):
    print(f'{step:7d}   {loss_history[step]:23.6f}   {error_history[step]:8d}')

## 4. Examine what your network learned

### 4.1. Plot the loss and mistakes

**Exercise 6 — construct the diagnostic curves.** Use your histories and the
affine histories from Section 2 to complete the supplied figure layout:

- Left: empirical logistic loss against completed updates, for both models.
- Middle: affine loss divided by $\log 2$, alongside its mistake count.
- Right: the same two quantities for your neural network.

Use `tab:purple` for the network loss and scaled losses, `tab:green` for mistakes,
and a dashed grey line for the affine loss in the first panel. Add labels and
legends. These are the same plotting operations used in Lab 4 and Section 2.

The scaling is for display only: each mistake contributes at least $\log 2$ to
the summed logistic loss, so mistakes cannot exceed loss divided by $\log 2$.
**The fitting updates used the unscaled empirical loss.**

In [ ]:
updates = range(num_updates + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
# YOUR CODE: draw the requested curves on axes[0], axes[1] and axes[2].

for ax in axes:
    ax.set_xlabel('Completed updates')
    ax.grid(alpha=0.2)
plt.show()

<details>
<summary>Show answer — loss and mistake curves</summary>

Both models use the same observations and loss. Use the same vertical range in
the two scaled-loss panels so their magnitudes are easy to compare. The histories
include update 0 as well as all 1,000 completed updates.

```python
updates = range(num_updates + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
axes[0].plot(updates, affine_loss_history, '--', color='0.4', label='Affine score')
axes[0].plot(updates, loss_history, color='tab:purple', label='Neural network')
axes[0].set_title('Empirical logistic loss')
axes[0].set_ylabel('Summed loss')
axes[0].legend()

for ax, losses, errors, title in (
    (axes[1], affine_loss_history, affine_error_history, 'Affine model'),
    (axes[2], loss_history, error_history, 'Neural network')
):
    ax.plot(updates, [value / math.log(2) for value in losses],
            color='tab:purple', label=r'Loss / $\log 2$')
    ax.plot(updates, errors, color='tab:green', label='Mistakes')
    ax.set_title(title + ': loss and mistakes')
    ax.set_ylabel('Value')
    ax.set_ylim(0, 235)
    ax.legend()
for ax in axes:
    ax.set_xlabel('Completed updates')
    ax.grid(alpha=0.2)
plt.show()

print('After 1,000 updates, on the same 100 observations:')
print(f'Affine model: loss = {affine_loss_history[-1]:.3f}, mistakes = {affine_error_history[-1]}')
print(f'Neural network: loss = {loss_history[-1]:.3f}, mistakes = {error_history[-1]}')
```

</details>

### 4.2. Inspect the decision boundaries

A neural network's zero-score set need not be a single straight line. The supplied
helper below draws it by evaluating scores on a grid and tracing the zero contour.
The black line separates light blue class-0 regions from light orange class-1
regions. At exactly zero, our classifier selects class 0.

Run this helper definition; reproducing its contour code is not an exercise.
`draw_boundary(ax, score_grid, title)` needs the axes on which to draw, the matrix
of grid scores, and a title. You will choose which saved network scores it displays.

In [ ]:
def draw_boundary(ax, score_grid, title):
    # Non-examinable helper: shade predicted classes and draw the score = 0 contour.
    classes = (score_grid > 0).to(t.float32)
    ax.contourf(x1_grid, x2_grid, classes,
                levels=[-0.5, 0.5, 1.5], colors=['#edf3fb', '#fff1e9'])
    ax.contour(x1_grid, x2_grid, score_grid, levels=[0], colors='black', linewidths=1)
    class_0 = y[:, 0] == 0
    class_1 = y[:, 0] == 1
    ax.scatter(X[class_0, 0], X[class_0, 1], color='tab:blue', s=18, label='Class 0')
    ax.scatter(X[class_1, 0], X[class_1, 1], color='tab:orange', marker='^', s=20, label='Class 1')
    ax.set(xlim=x1_limits, ylim=x2_limits, xlabel='$x_1$', ylabel='$x_2$', title=title)
    ax.set_aspect('equal')

**Exercise 7 — use the helper to compare the fitted boundaries.**

The cell below creates four panels and calculates the fitted affine scores on the
grid. Call `draw_boundary` on the first panel for those scores. On the remaining
panels, show your saved NN scores after 1, 100 and 1,000 updates, using
`boundary_scores[step]`. Include the corresponding mistake count in each title.

Why would calling `net(X_grid)` three times here fail to recover those three
earlier stages? Which array holds the mistake count for each completed update?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4), constrained_layout=True)
affine_grid = t.cat([t.ones(X_grid.shape[0], 1), X_grid], dim=1)
affine_grid_scores = (affine_grid @ affine_theta).reshape(x1_grid.shape)
# YOUR CODE: draw the affine result and the three saved NN snapshots.
plt.show()

<details>
<summary>Show answer — use the saved stages</summary>

The parameters currently contain the final fit: calling `net` again would show
that final model every time. `boundary_scores` holds the independently saved
values at each selected update; `error_history[step]` holds its mistake count.

```python
fig, axes = plt.subplots(1, 4, figsize=(15, 4), constrained_layout=True)
affine_grid = t.cat([t.ones(X_grid.shape[0], 1), X_grid], dim=1)
affine_grid_scores = (affine_grid @ affine_theta).reshape(x1_grid.shape)
draw_boundary(axes[0], affine_grid_scores,
              f'Affine: {num_updates} updates\n{affine_error_history[-1]} mistakes')
for ax, step in zip(axes[1:], (1, 100, num_updates)):
    draw_boundary(ax, boundary_scores[step],
                  f'NN: {step} update' + ('s' if step != 1 else '')
                  + f'\n{error_history[step]} mistakes')
axes[0].legend(loc='upper left', fontsize=8)
plt.show()
```

</details>

### 4.3. Check that the feature values changed

**Exercise 8 — inspect the learned representation of one fixed input.**

Keep the input row `x = X[:1]` fixed. Using the hidden-layer formula from Section
3.1, compute its hidden activations before fitting (from `initial_W1` and
`initial_b1`) and after fitting (from the current `W1` and `b1`). Use `no_grad`
because this is an inspection, not a derivative calculation.

Print the first five entries of each row and the largest absolute change across
all 100 entries. You can use `t.abs` and `.max()` for the latter. Did the input
change? Which parameters are responsible for the changed feature values? Would
adjusting only `W2` and `b2` have changed these hidden activations?

In [ ]:
# YOUR CODE: compare the same input's hidden activations before and after fitting.

<details>
<summary>Show answer — the same input, changed features</summary>

The input row is identical in both calculations. Changing `W1` and `b1` changes
the feature functions and hence can change their values at that input. The output
parameters combine these values into scores; changing only `W2` and `b2` would
leave the hidden activations unchanged. We are inspecting one row here, not
claiming every feature changes at every input.

```python
x = X[:1]
with t.no_grad():
    features_before = t.relu(x @ initial_W1 + initial_b1)
    features_after = t.relu(x @ W1 + b1)
print('Before, first five:', features_before[0, :5])
print('After, first five: ', features_after[0, :5])
print('Largest absolute change:', t.abs(features_after - features_before).max().item())
```

</details>

### Pause and interpret: what did the network add?

1. Compare the final empirical loss and mistake count of the two models. Which changes in the boundary help the network classify the four clouds?
2. Which parameters change the feature functions, and which parameters combine their values into an output score? Did we supply the XOR feature functions ourselves?
3. Does every decrease in loss reduce the mistake count? Inspect `loss_history[81]`, `loss_history[82]`, `error_history[81]` and `error_history[82]`.
4. What do these results tell us about fitting these observations? Do they establish the best possible network fit or its accuracy on unseen observations?

Write down your answers before opening the explanation.

<details>
<summary>Show answer — comparing the affine model and the network</summary>

**1. The network fits these observations better in this run.** After 1,000 updates, the affine model has loss about $69.308$ and makes 47 mistakes. The network has loss about $8.444$ and makes 3 mistakes. It makes 30 mistakes initially and after the first update, 7 after 100 updates, and 3 after 1,000. Its boundary changes direction and creates regions that follow the four-cloud arrangement more closely.

**2. The hidden parameters determine the features.** `W1` and `b1` define the functions whose values are stored in `phi1`. The output parameters `W2` and `b2` combine those activations into scores. We supplied the two raw coordinates and the network architecture; gradient descent adjusted all four parameter tensors. We did not specify the XOR feature functions by hand.

**3. Loss reduction and mistake reduction are different.** Between updates 81 and 82, the loss decreases from about $20.514$ to $20.398$, while the number of mistakes increases from 7 to 8. The loss depends on probabilities for all observations, whereas mistakes depend only on the thresholded predictions. The scaled-loss bound still holds at both steps.

**4. This is evidence about the fitted models on the observed data.** It shows that this network and fitting procedure achieve a smaller empirical loss and fewer mistakes here. It does not show that these are globally optimal network parameters, or measure prediction accuracy on unseen observations.


The numerical values above refer to the supplied data, initialisation and update
settings. Rerunning the fitting exercise restores the initial parameters. Rerunning
Exercise 1 also reproduces the initial draws by restoring the saved random state.
Numerical results can vary slightly across PyTorch versions and devices.

</details>

## 5. Optional extension: what if we remove ReLU?

**Exercise 9 — connect the calculation back to affine features.**

Without changing the stored parameters, calculate scores with the hidden ReLU
replaced by the identity. Expand the resulting expression algebraically: can you
write it as `X @ W_effective + b_effective`? Determine these effective coefficients
and check both calculations agree using `t.allclose(..., atol=1e-4, rtol=1e-4)`.

Use a separate expression, not a redefinition of `net`, and use `no_grad` for this
comparison. What does the identity tell you about the possible decision boundaries
of this model without ReLU, however its parameters are chosen?

In [ ]:
# OPTIONAL: derive the effective affine coefficients and check your calculation.

<details>
<summary>Show answer — composition without ReLU</summary>

Expanding gives
$$ (XW_1+b_1)W_2+b_2 = X(W_1W_2)+(b_1W_2+b_2). $$
The score is affine in the original inputs, with a straight zero-score boundary
when its input coefficients are nonzero. Extra hidden units do not remove that
restriction without a nonlinearity. The tolerance permits floating-point rounding.
This identity does not say that fitting this parameterisation would follow the
same update trajectory as our affine baseline.

```python
with t.no_grad():
    scores_without_relu = (X @ W1 + b1) @ W2 + b2
    W_effective = W1 @ W2
    b_effective = b1 @ W2 + b2
    effective_scores = X @ W_effective + b_effective
print('Same scores:', t.allclose(scores_without_relu, effective_scores,
                                atol=1e-4, rtol=1e-4))
```

</details>

You have translated a network architecture into tensors, implemented its score
function, calculated empirical losses and mistakes, obtained derivatives, and
used them in your own gradient-descent loop. Your curves and boundary snapshots
let you compare the fitted network with the affine model on the same data. You
also inspected how changing hidden parameters changes the feature representation.

This completes the neural-network fitting experiment.